# opentargets-py — Demo Notebook

Bu notebook, `opentargets-py` kütüphanesinin temel özelliklerini canlı API üzerinde gösteriyor.

**Kapsanan konular:**
1. Kurulum & import
2. Target (gen/protein) sorgulama
3. Hastalık sorgulama
4. İlaç sorgulama
5. Arama
6. Target → Hastalık ilişkileri (associations)
7. Toplu sorgu (batch)
8. Pandas DataFrame çıktısı
9. Cache ve performans

> API: https://api.platform.opentargets.org/api/v4/graphql

In [1]:
# Gerekli paketleri kur (ilk çalıştırmada)
# !pip install opentargets-py pandas

from opentargets import OpenTargetsClient

# Client oluştur — cache varsayılan olarak açık (5 dk TTL)
client = OpenTargetsClient()
print("✓ Client hazır")

✓ Client hazır


## 1. Target Sorgulama

Gen sembolü (`EGFR`) veya Ensembl ID (`ENSG00000146648`) ile sorgulayabilirsin.  
Kütüphane sembolü otomatik olarak Ensembl ID'ye çevirir ve cache'ler.

In [3]:
# Sembol ile sorgula
target = client.get_target("EGFR")

print(f"ID          : {target.id}")
print(f"Sembol      : {target.approved_symbol}")
print(f"Tam adı     : {target.approved_name}")
print(f"Biotype     : {target.biotype}")
if target.function_descriptions:
    print(f"Fonksiyon   : {target.function_descriptions[0][:500]}...")

ID          : ENSG00000146648
Sembol      : EGFR
Tam adı     : epidermal growth factor receptor
Biotype     : protein_coding
Fonksiyon   : Receptor tyrosine kinase binding ligands of the EGF family and activating several signaling cascades to convert extracellular cues into appropriate cellular responses (PubMed:10805725, PubMed:27153536, PubMed:2790960, PubMed:35538033). Known ligands include EGF, TGFA/TGF- alpha, AREG, epigen/EPGN, BTC/betacellulin, epiregulin/EREG and HBEGF/heparin-binding EGF (PubMed:12297049, PubMed:15611079, PubMed:17909029, PubMed:20837704, PubMed:27153536, PubMed:2790960, PubMed:7679104, PubMed:8144591, Pub...


In [4]:
# Ensembl ID ile direkt sorgula (symbol resolution atlanır, daha hızlı)
target2 = client.get_target("ENSG00000157764")  # BRAF
print(f"{target2.approved_symbol} — {target2.approved_name}")

BRAF — B-Raf proto-oncogene, serine/threonine kinase


## 2. Hastalık Sorgulama

EFO identifier ile hastalık bilgisi çek.

In [5]:
disease = client.get_disease("EFO_0003060")  # lung carcinoma

print(f"ID           : {disease.id}")
print(f"Ad           : {disease.name}")
print(f"Açıklama     : {disease.description[:150]}...")
print(f"Terapötik alan: {', '.join(disease.therapeutic_areas[:3])}")

ID           : EFO_0003060
Ad           : non-small cell lung carcinoma
Açıklama     : A group of at least three distinct histological types of lung cancer, including non-small cell squamous cell carcinoma, adenocarcinoma, and large cell...
Terapötik alan: respiratory or thoracic disease, cancer or benign tumor


## 3. İlaç Sorgulama

In [6]:
drug = client.get_drug("CHEMBL939")  # Erlotinib

print(f"ID           : {drug.id}")
print(f"Ad           : {drug.name}")
print(f"Tür          : {drug.drug_type}")
print(f"Etki mekanizması: {drug.mechanism_of_action}")
print(f"Klinik faz   : {drug.max_clinical_trial_phase}")
print(f"Ticari adlar : {', '.join(drug.trade_names)}")

ID           : CHEMBL939
Ad           : GEFITINIB
Tür          : Small molecule
Etki mekanizması: Epidermal growth factor receptor erbB1 inhibitor
Klinik faz   : APPROVAL
Ticari adlar : Gefitinib mylan, Iressa


In [7]:
# İlacın endikasyonları (hangi hastalıklar için onaylı)
indications = client.get_drug_indications("CHEMBL939")
print(f"Erlotinib endikasyonları ({len(indications)} adet):\n")
for ind in indications[:5]:
    print(f"  • {ind.disease_name:<45} faz={ind.max_phase_for_indication}")

Erlotinib endikasyonları (37 adet):

  • germ cell tumor                               faz=PHASE_2
  • nasopharyngeal carcinoma                      faz=PHASE_1_2
  • renal cell carcinoma                          faz=PHASE_1_2
  • metastatic prostate cancer                    faz=PHASE_1_2
  • small cell lung carcinoma                     faz=PHASE_2


## 4. Arama (Search)

Serbest metin ile target, hastalık veya ilaç ara.

In [8]:
# Tüm entity tiplerinde ara
results = client.search("BRCA1", limit=5)
print("'BRCA1' için arama sonuçları:\n")
for r in results:
    print(f"  [{r.entity_type:<8}] {r.name:<30} {r.id}")

'BRCA1' için arama sonuçları:

  [target  ] BRCA1                          ENSG00000012048
  [disease ] BRCA1-related cancer predisposition MONDO_0700268
  [disease ] pancreatic cancer, susceptibility to, 4 MONDO_0013685
  [disease ] hereditary breast ovarian cancer syndrome MONDO_0003582
  [disease ] Hereditary breast and ovarian cancer syndrome Orphanet_145


In [9]:
# Sadece hastalık ara
disease_results = client.search("breast cancer", entity_type="disease", limit=5)
print("'breast cancer' hastalık araması:\n")
for r in disease_results:
    print(f"  {r.name:<45} {r.id}")

'breast cancer' hastalık araması:

  breast cancer                                 MONDO_0007254
  breast carcinoma                              EFO_0000305
  Hereditary breast and ovarian cancer syndrome Orphanet_145
  hereditary breast ovarian cancer syndrome     MONDO_0003582
  Hereditary breast cancer                      Orphanet_227535


## 5. Target → Hastalık İlişkileri (Associations)

Bir genin hangi hastalıklarla ilişkili olduğunu, kanıt skorlarıyla birlikte çek.

In [10]:
associations = client.get_target_associations("EGFR", limit=10)

print(f"EGFR ile ilişkili ilk {len(associations)} hastalık:\n")
print(f"{'Hastalık':<45} {'Skor':>6}  {'Datasource sayısı':>18}")
print("-" * 75)
for a in associations:
    n_ds = len([ds for ds in a.datasource_scores if ds.score > 0])
    print(f"{a.disease_name:<45} {a.score:>6.3f}  {n_ds:>18}")

EGFR ile ilişkili ilk 10 hastalık:

Hastalık                                        Skor   Datasource sayısı
---------------------------------------------------------------------------
non-small cell lung carcinoma                  0.852                   9
lung cancer                                    0.766                   6
lung adenocarcinoma                            0.764                   7
cancer                                         0.746                   3
breast cancer                                  0.678                   3
colorectal adenocarcinoma                      0.670                   6
head and neck squamous cell carcinoma          0.669                   6
head and neck malignant neoplasia              0.655                   3
neoplasm                                       0.655                   3
glioblastoma multiforme                        0.649                   6


In [10]:
# Belirli bir target–disease çifti için skor
assoc = client.get_associations(
    target_id="ENSG00000146648",  # EGFR
    disease_id="EFO_0003060",     # lung carcinoma
)

if assoc:
    print(f"EGFR ↔ Lung carcinoma skoru: {assoc.score:.4f}")
    print("\nDatasource bazlı skorlar:")
    for ds in sorted(assoc.datasource_scores, key=lambda x: x.score, reverse=True):
        bar = "█" * int(ds.score * 20)
        print(f"  {ds.id:<35} {ds.score:.3f} {bar}")

EGFR ↔ Lung carcinoma skoru: 0.8524

Datasource bazlı skorlar:
  europepmc                           0.999 ███████████████████
  clinical_precedence                 0.996 ███████████████████
  cancer_biomarkers                   0.963 ███████████████████
  eva_somatic                         0.882 █████████████████
  eva                                 0.746 ██████████████
  intogen                             0.731 ██████████████
  cancer_gene_census                  0.608 ████████████
  clingen                             0.608 ████████████
  crispr                              0.252 █████


## 6. Toplu Sorgu (Batch)

Birden fazla geni tek API çağrısında çek.

In [11]:
import time

# Tek tek çekme — 4 ayrı API çağrısı
t0 = time.time()
targets_one_by_one = [client.get_target(g) for g in ["EGFR", "BRAF", "KRAS", "TP53"]]
t1 = time.time()

# Toplu çekme — 1 API çağrısı
targets_batch = client.get_targets(["EGFR", "BRAF", "KRAS", "TP53"])
t2 = time.time()

print(f"Tek tek : {t1-t0:.2f}s  (cache'den geldiği için hızlı görünür)")
print(f"Batch   : {t2-t1:.2f}s\n")

print("Batch sonuçları:")
for t in targets_batch:
    print(f"  {t.approved_symbol:<8} {t.id}  —  {t.biotype}")

Tek tek : 0.56s  (cache'den geldiği için hızlı görünür)
Batch   : 0.10s

Batch sonuçları:
  EGFR     ENSG00000146648  —  protein_coding
  BRAF     ENSG00000157764  —  protein_coding
  KRAS     ENSG00000133703  —  protein_coding
  TP53     ENSG00000141510  —  protein_coding


## 7. Pandas DataFrame Çıktısı

`as_dataframe=True` ile herhangi bir liste metodundan direkt DataFrame al.

In [12]:
df = client.get_target_associations("EGFR", limit=50, as_dataframe=True)

print(f"DataFrame boyutu: {df.shape}\n")
print(df[["disease_name", "score", "evidence_count"]].head(10).to_string(index=False))

DataFrame boyutu: (50, 7)

                         disease_name    score  evidence_count
        non-small cell lung carcinoma 0.852433               0
                          lung cancer 0.766258               0
                  lung adenocarcinoma 0.763945               0
                               cancer 0.745873               0
                        breast cancer 0.678243               0
            colorectal adenocarcinoma 0.669844               0
head and neck squamous cell carcinoma 0.668905               0
    head and neck malignant neoplasia 0.654672               0
                             neoplasm 0.654535               0
              glioblastoma multiforme 0.648555               0


In [13]:
# Skora göre filtrele ve sırala
top = df[df["score"] >= 0.5].sort_values("score", ascending=False)
print(f"Skor ≥ 0.5 olan hastalıklar: {len(top)}\n")
print(top[["disease_name", "score"]].head(10).to_string(index=False))

Skor ≥ 0.5 olan hastalıklar: 25

                         disease_name    score
        non-small cell lung carcinoma 0.852433
                          lung cancer 0.766258
                  lung adenocarcinoma 0.763945
                               cancer 0.745873
                        breast cancer 0.678243
            colorectal adenocarcinoma 0.669844
head and neck squamous cell carcinoma 0.668905
    head and neck malignant neoplasia 0.654672
                             neoplasm 0.654535
              glioblastoma multiforme 0.648555


## 8. Hata Yönetimi

Bulunamayan entity'ler için `NotFoundError` fırlatılır.

In [14]:
from opentargets import NotFoundError, APIError

# Var olmayan bir gen
try:
    client.get_target("GENEBULUNAMADI123")
except NotFoundError as e:
    print(f"NotFoundError: entity_type={e.entity_type!r}, entity_id={e.entity_id!r}")

# Var olmayan bir hastalık
try:
    client.get_disease("EFO_9999999")
except NotFoundError as e:
    print(f"NotFoundError: {e}")

NotFoundError: entity_type='target', entity_id='GENEBULUNAMADI123'


NotFoundError: disease not found: 'EFO_9999999'


## 9. Cache Performansı

Aynı sorgu ikinci kez çekildiğinde cache'den gelir, API'ye istek atmaz.

In [11]:
fresh_client = OpenTargetsClient(cache=True)

# İlk istek — API'ye gider
t0 = time.time()
_ = fresh_client.get_target("EGFR")
t1 = time.time()
print(f"1. istek (API): {(t1-t0)*1000:.0f} ms")

# İkinci istek — cache'den gelir
t2 = time.time()
_ = fresh_client.get_target("EGFR")
t3 = time.time()
print(f"2. istek (cache): {(t3-t2)*1000:.1f} ms")

# Cache kapalı
no_cache = OpenTargetsClient(cache=False)
t4 = time.time()
_ = no_cache.get_target("EGFR")
t5 = time.time()
print(f"Cache=False (her zaman API): {(t5-t4)*1000:.0f} ms")

NameError: name 'time' is not defined

## 10. Context Manager ile Temiz Kullanım

`with` bloğu çıkışında HTTP bağlantıları otomatik kapatılır.

In [16]:
with OpenTargetsClient() as c:
    tp53 = c.get_target("TP53")
    print(f"{tp53.approved_symbol}: {tp53.approved_name}")
    
    drugs = c.get_target_drugs("TP53")
    print(f"TP53 için bilinen ilaç sayısı: {len(drugs)}")

print("✓ Bağlantı kapatıldı")

TP53: tumor protein p53
TP53 için bilinen ilaç sayısı: 8
✓ Bağlantı kapatıldı
